# SIM 1000s — Results File Availability Check

Scans `I001_Results/` (pickles + CSVs), `I001_Results/OBJ_files/` (job metadata),
and `E001_Simulations/` (Abaqus `.inp` decks and `.odb` result databases) for a range of simulation numbers
and reports, per simulation, which result files exist and whether their size
clears a "looks like real data" threshold.

**Why a size threshold?** A pickle/CSV can exist on disk but be empty or
truncated (crashed reduction step, interrupted write, etc.). Flagging files
below a minimum size surfaces those cases instead of just checking existence.

**Caveat:** the blanket 100 KB threshold the size check defaults to is right
for the "full field" pickles (`A`, `B`, `C`, `C2`, `TP1`, `TP2`, `I1_*`,
`I2_*`, `DEFC1`) which are always MB-to-GB when valid. A few pickles are
*light summaries* by design and are legitimately a few KB even when perfectly
valid (`A2`, `DEFC2`, `E`, `EIG`, `I3_BFS_*`, `TP2_L`) or tiny metadata files
(`CSV`, `OBJ`). Those get their own lower threshold in `MIN_SIZE_OVERRIDES`
below — edit it if a kind is still misclassified.

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# -----------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------
ROOT         = Path('..')
RESULTS_DIR  = ROOT / 'I001_Results'
OBJ_DIR      = RESULTS_DIR / 'OBJ_files'
SIM_DIR      = ROOT / 'E001_Simulations'

SIM_START = 1000
SIM_END   = 1090   # inclusive
SIMS      = list(range(SIM_START, SIM_END + 1))

# Default "looks like real data" floor for any file kind not listed below.
MIN_SIZE_BYTES = 100_000  # 100 kB

# Per-kind overrides for files that are legitimately small even when valid
# (light/summary pickles and small metadata files) — see markdown above.
MIN_SIZE_OVERRIDES = {
    'A2':          1_000,   # RF2/U2/t history only (~10 KB typical)
    'DEFC2':       1_000,   # localization scalars per frame (~20 KB typical)
    'E':             200,   # lambda_min time series (~KB typical; 0 rows -> ~100 B)
    'EIG':            10,   # tiny json list; empty -> 2 bytes ('[]')
    'I3_BFS_3002':  1_000,  # global scalar time series (~9 KB typical)
    'TP2_L':      10_000,   # 'light' statistical summary (~50 KB typical)
    'CSV':            10,   # small element-set/result summary file
    'OBJ':         1_000,   # job metadata json (~5-10 KB typical)
}

# Canonical set of pickle "kinds" produced by the reduction pipeline for this
# series (see A001_functions/Reduce_resultsV5.py). Always shown as columns
# even if no simulation in the range has produced them yet.
CORE_PKL_KINDS = [
    'A', 'A2', 'B', 'C', 'C2', 'D', 'E', 'EIG',
    'T1', 'T2', 'DEFC1', 'DEFC2',
    'I1_3002', 'I2_3002', 'I3_BFS_3002',
    'TP1', 'TP2', 'TP2_L',
]
EXTRA_KINDS = ['CSV', 'OBJ', 'INP', 'ODB']


## Scan disk for what actually exists

In [ ]:
PKL_RE = re.compile(r'^DATA_PICK_(\d+)_(.+)\.(pkl|json)$')

def scan_pickles():
    """One pass over I001_Results/, mapping sim -> {kind: size_bytes}."""
    per_sim = {sim: {} for sim in SIMS}
    sim_set = set(SIMS)
    with os.scandir(RESULTS_DIR) as it:
        for entry in it:
            if not entry.is_file():
                continue
            m = PKL_RE.match(entry.name)
            if not m:
                continue
            sim = int(m.group(1))
            if sim not in sim_set:
                continue
            kind = m.group(2)
            per_sim[sim][kind] = entry.stat().st_size
    return per_sim

def scan_csv():
    per_sim = {}
    with os.scandir(RESULTS_DIR) as it:
        for entry in it:
            if not entry.is_file() or not entry.name.startswith('RES_SIM_'):
                continue
            m = re.match(r'^RES_SIM_(\d+)\.csv$', entry.name)
            if not m:
                continue
            sim = int(m.group(1))
            if sim in SIMS:
                per_sim[sim] = entry.stat().st_size
    return per_sim

def scan_obj():
    per_sim = {}
    for sim in SIMS:
        p = OBJ_DIR / f'SIM_{sim:03d}.json'
        if p.exists():
            per_sim[sim] = p.stat().st_size
    return per_sim

def scan_inp():
    per_sim = {}
    for sim in SIMS:
        p = SIM_DIR / f'SIM_{sim:03d}' / f'SIM_{sim:03d}.inp'
        if p.exists():
            per_sim[sim] = p.stat().st_size
    return per_sim

def scan_odb():
    """Main analysis job output database (E001_Simulations/SIM_x/SIM_x.odb).
    Note: does not track the separate *_EIGJOB.odb used by the standalone
    stiffness-eigenvalue replay job."""
    per_sim = {}
    for sim in SIMS:
        p = SIM_DIR / f'SIM_{sim:03d}' / f'SIM_{sim:03d}.odb'
        if p.exists():
            per_sim[sim] = p.stat().st_size
    return per_sim

pkl_sizes = scan_pickles()
csv_sizes = scan_csv()
obj_sizes = scan_obj()
inp_sizes = scan_inp()
odb_sizes = scan_odb()

# Kinds actually observed on disk (may include ones not in CORE_PKL_KINDS,
# e.g. future stages like J1/J2/H1/H2/K1/K2/Q1/Q2).
observed_kinds = sorted({kind for d in pkl_sizes.values() for kind in d})
ALL_KINDS = sorted(set(CORE_PKL_KINDS) | set(observed_kinds)) + EXTRA_KINDS
print(f'{len(SIMS)} simulations scanned ({SIM_START}-{SIM_END})')
print(f'{len(ALL_KINDS)} file kinds tracked: {ALL_KINDS}')


91 simulations scanned (1000-1090)
22 file kinds tracked: ['A', 'A2', 'B', 'C', 'C2', 'D', 'DEFC1', 'DEFC2', 'E', 'EIG', 'I1_3002', 'I2_3002', 'I3_BFS_3002', 'T1', 'T2', 'TP1', 'TP2', 'TP2_L', 'CSV', 'OBJ', 'INP', 'ODB']


## Build the size / status tables

In [ ]:
def size_for(sim, kind):
    if kind == 'CSV':
        return csv_sizes.get(sim)
    if kind == 'OBJ':
        return obj_sizes.get(sim)
    if kind == 'INP':
        return inp_sizes.get(sim)
    if kind == 'ODB':
        return odb_sizes.get(sim)
    return pkl_sizes.get(sim, {}).get(kind)

def classify(kind, size):
    if size is None:
        return 'missing'
    thresh = MIN_SIZE_OVERRIDES.get(kind, MIN_SIZE_BYTES)
    return 'valid' if size >= thresh else 'invalid'

size_kb_df = pd.DataFrame(
    {kind: [ (size_for(sim, kind) / 1024) if size_for(sim, kind) is not None else np.nan
             for sim in SIMS ]
     for kind in ALL_KINDS},
    index=SIMS,
)
size_kb_df.index.name = 'SIM'

status_df = pd.DataFrame(
    {kind: [classify(kind, size_for(sim, kind)) for sim in SIMS] for kind in ALL_KINDS},
    index=SIMS,
)
status_df.index.name = 'SIM'

status_df.head()


,A,A2,B,C,C2,D,DEFC1,DEFC2,E,EIG,I1_3002,I2_3002,I3_BFS_3002,T1,T2,TP1,TP2,TP2_L,CSV,OBJ,INP,ODB
SIM,,,,,,,,,,,,,,,,,,,,,,
1000,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,missing,valid,valid,missing
1001,valid,valid,valid,valid,valid,missing,valid,valid,invalid,invalid,valid,valid,valid,missing,missing,valid,valid,valid,valid,valid,valid,missing
1002,valid,valid,valid,valid,valid,missing,valid,valid,valid,valid,valid,valid,valid,missing,missing,valid,valid,valid,valid,valid,valid,missing
1003,valid,valid,valid,valid,valid,missing,valid,valid,missing,missing,valid,valid,valid,missing,missing,valid,valid,valid,valid,valid,valid,missing
1004,valid,valid,valid,valid,valid,missing,valid,valid,missing,missing,valid,valid,valid,missing,missing,valid,valid,valid,valid,valid,valid,missing


## Styled table — green = valid, red = present but suspiciously small, grey = missing

In [ ]:
STATUS_COLORS = {
    'valid':   'background-color:#c6efce; color:#006100',
    'invalid': 'background-color:#ffc7ce; color:#9c0006',
    'missing': 'background-color:#f2f2f2; color:#aaaaaa',
}

def _tooltip_text(sim, kind, size, status):
    size_txt = '-' if pd.isna(size) else f'{size:,.1f} KB'
    return f'SIM {sim}  |  {kind}  |  {status}  |  {size_txt}'

def style_size_table(size_df, status_df):
    def _paint(_):
        return pd.DataFrame(
            [[STATUS_COLORS[status_df.loc[i, c]] for c in size_df.columns] for i in size_df.index],
            index=size_df.index, columns=size_df.columns,
        )
    tooltips = pd.DataFrame(
        [[_tooltip_text(sim, kind, size_df.loc[sim, kind], status_df.loc[sim, kind])
          for kind in size_df.columns] for sim in size_df.index],
        index=size_df.index, columns=size_df.columns,
    )
    return (
        size_df.style
        .apply(_paint, axis=None)
        .format(lambda v: '-' if pd.isna(v) else (f'{v:,.0f}' if v >= 10 else f'{v:.2f}'))
        .set_caption('File size (KB) per SIM x kind — colored by validity (hover a cell for details)')
        .set_tooltips(tooltips, css_class='sim-tooltip')
        .set_table_styles([
            {'selector': '.sim-tooltip',
             'props': [('visibility', 'hidden'), ('position', 'absolute'), ('z-index', '1'),
                       ('background-color', '#333'), ('color', 'white'),
                       ('font-size', '0.75em'), ('padding', '3px 6px'), ('border-radius', '3px'),
                       ('transform', 'translate(-20%, -100%)')]},
            {'selector': 'td:hover .sim-tooltip', 'props': [('visibility', 'visible')]},
        ], overwrite=False)
    )

style_size_table(size_kb_df, status_df)


## Per-simulation completeness summary

In [ ]:
summary_rows = []
for sim in SIMS:
    counts = status_df.loc[sim].value_counts()
    n_valid   = int(counts.get('valid', 0))
    n_invalid = int(counts.get('invalid', 0))
    n_missing = int(counts.get('missing', 0))
    missing_or_bad = [k for k in ALL_KINDS if status_df.loc[sim, k] != 'valid']
    summary_rows.append(dict(
        SIM=sim,
        valid=n_valid,
        invalid=n_invalid,
        missing=n_missing,
        complete=(n_invalid == 0 and n_missing == 0),
        problem_kinds=', '.join(missing_or_bad) if missing_or_bad else '',
    ))

summary_df = pd.DataFrame(summary_rows).set_index('SIM')
print(f"Fully complete simulations: {int(summary_df['complete'].sum())} / {len(SIMS)}")
summary_df


Fully complete simulations: 0 / 91


,valid,invalid,missing,complete,problem_kinds
SIM,,,,,
1000,2,0,20,False,"A, A2, B, C, C2, D, DEFC1, DEFC2, E, EIG, I1_3..."
1001,16,2,4,False,"D, E, EIG, T1, T2, ODB"
1002,18,0,4,False,"D, T1, T2, ODB"
1003,16,0,6,False,"D, E, EIG, T1, T2, ODB"
1004,16,0,6,False,"D, E, EIG, T1, T2, ODB"
...,...,...,...,...,...
1086,2,0,20,False,"A, A2, B, C, C2, D, DEFC1, DEFC2, E, EIG, I1_3..."
1087,2,0,20,False,"A, A2, B, C, C2, D, DEFC1, DEFC2, E, EIG, I1_3..."
1088,2,0,20,False,"A, A2, B, C, C2, D, DEFC1, DEFC2, E, EIG, I1_3..."


## Interactive heatmap (hover any cell to see SIM, kind, status, size)

In [ ]:
import plotly.graph_objects as go

status_map = {'missing': 0, 'invalid': 1, 'valid': 2}
status_code = status_df.apply(lambda col: col.map(status_map)).astype(int)

# customdata carries (status, size_kb) per cell so the hover text can show
# the exact simulation, file kind, status, and size on mouseover.
customdata = np.dstack([
    status_df.values,
    size_kb_df.values,
])

fig = go.Figure(data=go.Heatmap(
    z=status_code.values,
    x=ALL_KINDS,
    y=[str(s) for s in SIMS],
    customdata=customdata,
    colorscale=[[0.0, '#f2f2f2'], [0.33, '#f2f2f2'],
                [0.34, '#ffc7ce'], [0.66, '#ffc7ce'],
                [0.67, '#c6efce'], [1.0, '#c6efce']],
    zmin=0, zmax=2,
    showscale=False,
    xgap=1, ygap=1,
    hovertemplate=(
        'SIM %{y}<br>'
        'kind: %{x}<br>'
        'status: %{customdata[0]}<br>'
        'size: %{customdata[1]:.1f} KB'
        '<extra></extra>'
    ),
))

fig.update_layout(
    title=f'Result availability — SIM {SIM_START}-{SIM_END} (hover a cell for details)',
    xaxis_title='File kind',
    yaxis_title='SIM',
    width=max(700, 40 * len(ALL_KINDS)),
    height=max(500, 16 * len(SIMS)),
    yaxis=dict(autorange='reversed', type='category', tickfont=dict(size=9)),
    xaxis=dict(type='category', tickangle=-90),
    margin=dict(l=60, r=20, t=60, b=100),
)
fig.show()


## Notes

- To check a different range, edit `SIM_START`/`SIM_END` in the config cell and re-run.
- To tighten/loosen the "valid" threshold for a given file kind, edit `MIN_SIZE_BYTES`
  (global default) or add/adjust an entry in `MIN_SIZE_OVERRIDES`.
- `problem_kinds` in the summary table lists every kind that is either missing or under
  its size threshold for that simulation — use it to find exactly what still needs to be
  (re-)run or (re-)reduced.
- New reduction stages (e.g. J/K/H/Q pickles) are picked up automatically: any kind seen
  on disk for at least one simulation in range is added as a column, even if not listed
  in `CORE_PKL_KINDS`.